In [ ]:
from pathlib import Path
import numpy as np
from Physical_Quantity.Physical_Data_Record import SuperCorrRecord
from Physical_Quantity.Super_Correlation_Process import SuperCorrProcessor
from Physical_Quantity.Super_Correlation_Analyzer import (
    SuperCorrDecayAnalyzer, AnalyzerConfig,
)

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-------------------- 真实数据路径一：N400 / t_J_3 / d1_10 / bond_dim 1500 / refbond 97_98 --------------------
### ================= 文件夹路径变量（需要修改的部分）=====================
N = 400
a = 1.0
t_J = "t_J_3"              # t/J
doping = "d1_10"           # 空穴率
bond_dim = "1500"          # bond dimension
version = "97_98"          # reference bond

base_dir = Path(f'Data_N{N}/{t_J}/{doping}/{bond_dim}_measure_super_corr/super_corr_{version}')

### 提取 + 合成（与 Superconductivity_Correlation.ipynb 相同流程）
records = {}
for bond in 'ABC':
    for channel in 'abcd':
        key = f'scs{bond}{channel}'
        records[key] = SuperCorrRecord.extract_data(
            (base_dir / f'{key}_fix_refbond_{version}.json').read_text(encoding='utf-8'), N, a)

scs_a, scs_b, scs_c, scs_d = records['scsAa'], records['scsAb'], records['scsAc'], records['scsAd']
err_ad, err_bc = SuperCorrProcessor.symmetry_check(scs_a, scs_b, scs_c, scs_d)
print(f'SU(2) 对称性: err_ad={err_ad:.3e}, err_bc={err_bc:.3e}（~1e-4 可接受，>1e-3 警惕）')
combined_A = SuperCorrProcessor.Combined_super_correlation(scs_a, scs_b, scs_c, scs_d)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
### 衰减分析（默认配置）
result_N400 = SuperCorrDecayAnalyzer.analyze(combined_A)
SuperCorrDecayAnalyzer.print_summary(result_N400)

### 综合 8 面板图（show=False 时静默出图，save_path 保存 PNG）
prep = SuperCorrDecayAnalyzer.group_by_distance(combined_A)
fig_N400 = SuperCorrDecayAnalyzer.Decay_Visualization(result_N400, prep.r, prep.C, show=False)
display(fig_N400)
plt.close(fig_N400)  # 关闭防 inline 后端重复渲染

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-------------------- 真实数据路径二：N96 / t_J_3 / d1_48 / refbond 25_26（无振荡对照） --------------------
N2 = 96
base2 = Path(f'Data_N{N2}/t_J_3/d1_48/measure_super_corr/super_corr_25_26')
recs2 = {}
for bond in 'ABC':
    for channel in 'abcd':
        key = f'scs{bond}{channel}'
        recs2[key] = SuperCorrRecord.extract_data(
            (base2 / f'{key}_fix_refbond_25_26.json').read_text(encoding='utf-8'), N2, 1.0)
combined_N96 = SuperCorrProcessor.Combined_super_correlation(
    recs2['scsAa'], recs2['scsAb'], recs2['scsAc'], recs2['scsAd'])

result_N96 = SuperCorrDecayAnalyzer.analyze(combined_N96)
SuperCorrDecayAnalyzer.print_summary(result_N96)

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-------------------- 合成数据分类测试（8 类 + 边界用例） --------------------
def synth(r, kind, A=1e-4, K=2.0, xi=15.0, Q=0.86, phi=0.3, noise_amp=0.05, seed=42):
    r = np.asarray(r, dtype=float)
    g = np.random.default_rng(seed)
    if kind == 'constant':   base = np.full_like(r, A)
    elif kind == 'power':    base = A * r ** (-K)
    elif kind == 'exp':      base = A * np.exp(-r / xi)
    elif kind == 'osc_power': base = A * r ** (-K) * np.cos(Q * r + phi)
    elif kind == 'osc_exp':  base = A * np.exp(-r / xi) * np.cos(Q * r + phi)
    else: raise ValueError(kind)
    return base + noise_amp * np.abs(base) * g.standard_normal(len(r))

def make_records(r, C):
    return [SuperCorrRecord((0, 1), (2, 3), float(d), float(v)) for d, v in zip(r, C)]

r_grid = np.arange(3.0, 76.0, 3.0)   # 与 N400 提取后距离网格一致（25 点）
for kind in ['constant', 'power', 'exp', 'osc_power', 'osc_exp']:
    res = SuperCorrDecayAnalyzer.analyze(make_records(r_grid, synth(r_grid, kind)))
    print(f'{kind:<10} -> {res.classification:<24} (evidence: {res.evidence_level})')

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-------------------- 批量分析（不同 refbond / doping；不自动做跨数据集物理结论） --------------------
config = AnalyzerConfig()
results = SuperCorrDecayAnalyzer.analyze_batch({
    'N400_97_98': combined_A,
    'N96_25_26':  combined_N96,
}, config)
for name, res in results.items():
    print(f'{name:<12} -> {res.classification:<24} Q={res.oscillation.Q:.4g} '
          f'K/xi={res.envelope_fit.params.get("K", res.envelope_fit.params.get("xi", float("nan"))):.4g} '
          f'evidence={res.evidence_level}')
fig_compare = SuperCorrDecayAnalyzer.Decay_Visualization_compare(results, show=False)
display(fig_compare)
plt.close(fig_compare)  # 关闭防 inline 后端重复渲染
